# Imports

In [ ]:
import os
import sys

In [ ]:
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [ ]:
sys.path.insert(0, f'{project_dir}/code/Packages')

In [ ]:
import json
import preprocessing
from tqdm import tqdm
import igraph as ig

In [ ]:
preprocessing.set_export_folder("final_edgelists")
nodelist = preprocessing.import_nodelist()

Export directory set to G:\Shared drives\DOST FilWordNet x WordSense\Thesis\NetSci Team\documents\THS3\Temp Final Deliverables\SOURCE/data/3 - Network Generation/final_edgelists.


In [ ]:
from wsi import community_detection as cd

---

# Import network

In [ ]:
graphml_file = 'semset_network'

In [ ]:
network = ig.read(f"{project_dir}/data/7 - Semset Creation/{graphml_file}.graphml", format="graphml")

In [ ]:
print(f'Number of nodes: {len(network.vs)}')

Number of nodes: 11548


In [ ]:
print(f'Number of edges: {len(network.es)}')

Number of edges: 66657106


# Edge Filtering

In [ ]:
temp_network = network.copy()

In [ ]:
print(f"{len(temp_network.es)} to ", end="")
temp_network.delete_edges([edge for edge in temp_network.es.select(weight_lt=0.3)])
print(len(temp_network.es))

19399355 to 8942


# Community Detection

In [ ]:
# comm detection
def comm_detection(network):
#     partitions = cd.get_node_comm(cd.leiden_cpm_algorithm(network, 0.2, True))
    partitions = cd.get_node_comm(cd.leiden_modularity_algorithm(network, True))
#     partitions = cd.get_node_comm(cd.louvain_algorithm(network, True))

    # converts the node ids into the sentence ids
    semsets = []
    for index, community in enumerate(partitions.values()):
        temp = []
        for node_index in community:
            temp.append(network.vs[node_index]['name'])
        semsets.append(temp)

    return semsets

In [ ]:
def print_semsets(semsets):
    
    print('\n')
    for index, comm in enumerate(semsets):
        print(f"Semset {index}:")
        print(comm)
        
        # for sense_id in comm:
        #   print(f"\t{sense_id}")
        
        #print('\n')

# Results

In [ ]:
comm_detection_results = comm_detection(temp_network)

In [ ]:
print_semsets(comm_detection_results)

In [ ]:
comm_detection_dict = {f'{index}': community for index, community in enumerate(comm_detection_results)}

In [ ]:
# if os.path.isdir(f"{project_dir}/data/7 - Semset Creation/semsets"):
#     os.makedirs(f"{project_dir}/data/7 - Semset Creation/semsets")

## Save as JSON

In [ ]:
with open(f"{project_dir}/data/7 - Semset Creation/semsets_leiden_mod_removed0.3.json", "w") as f:
    json.dump(comm_detection_dict, f)

## Save as excel sheet (for viewing)

In [ ]:
comm_detection_sr = pd.Series(comm_detection_dict)

In [ ]:
file_name = f"{project_dir}/data/7 - Semset Creation/semsets_leiden_mod_removed0.3.xlsx"
comm_detection_sr.to_excel(file_name)